<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW16 · Session 1 · Datetime Indexes, Lags, Rolling Measures, and Resampling

        **Course level:** developing beginner  
        **Prior learning:** pandas indexing, missing data, line charts, and basic growth calculations  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Time-series rows have an order that carries information. We create a reliable datetime index, identify gaps, compare current and past values, smooth variation, and change frequency.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - parse date strings and create a sorted DatetimeIndex
- detect duplicate and missing periods
- create lags, differences, and percentage changes
- calculate rolling summaries without using future values
- resample monthly data to quarterly and annual frequency


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load and inspect monthly observations

parse_dates converts the month column at import, but we still verify order and uniqueness.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

monthly = pd.read_csv(DATA_DIR / "monthly_sales.csv", parse_dates=["month"])
print(monthly.head())
print(monthly.dtypes)


## 1. Create a trustworthy time index

A time series should have an explicit datetime type, chronological order, a defined frequency, and a rule for duplicate periods.


### Sort and set the DatetimeIndex

The index is unique and monotonic after sorting.


In [ ]:
series = monthly.sort_values("month").set_index("month")

print("Index type:", type(series.index))
print("Sorted:", series.index.is_monotonic_increasing)
print("Unique:", series.index.is_unique)
print("Inferred frequency:", pd.infer_freq(series.index))


### Check for missing calendar periods

asfreq creates the complete expected monthly grid. Newly created rows would expose absent months.


In [ ]:
regular = series.asfreq("MS")
missing_period_rows = regular[regular[["sales", "orders", "marketing_spend"]].isna().all(axis=1)]

print("Expected months:", len(regular))
print("Completely absent months:", len(missing_period_rows))
print("Missing sales values:", regular["sales"].isna().sum())


## 2. Extract calendar information

Datetime indexes expose year, month, quarter, and day information. Calendar features support grouping, but avoid assuming a pattern will continue.


### Add readable calendar columns

The Period representation is useful for labels and grouping.


In [ ]:
calendar_view = regular.copy()
calendar_view["year"] = calendar_view.index.year
calendar_view["month_number"] = calendar_view.index.month
calendar_view["month_name"] = calendar_view.index.month_name()
calendar_view["quarter"] = calendar_view.index.to_period("Q").astype(str)

print(calendar_view.head())


## 3. Lag, difference, and percentage change

- shift(1) aligns the previous observation with the current row;
- diff() calculates current minus previous;
- pct_change() calculates proportional change.

The first value is missing because no earlier observation exists.


### Create one-month comparisons

fill_method=None prevents hidden filling across missing observations.


In [ ]:
changes = regular[["sales"]].copy()
changes["sales_lag_1"] = changes["sales"].shift(1)
changes["sales_difference"] = changes["sales"].diff(1)
changes["sales_growth_pct"] = changes["sales"].pct_change(fill_method=None) * 100

print(changes.head(8).round(2))


### Compare with the same month one year earlier

A 12-period lag is meaningful only after establishing monthly frequency.


In [ ]:
changes["sales_lag_12"] = changes["sales"].shift(12)
changes["year_on_year_growth_pct"] = changes["sales"].pct_change(
    periods=12,
    fill_method=None,
) * 100

print(changes[["sales", "sales_lag_12", "year_on_year_growth_pct"]].tail(15).round(2))


## 4. Rolling windows

A trailing rolling window summarises the current and preceding observations. It must not include future values when used for forecasting.


### Calculate trailing rolling statistics

min_periods controls how many observations are required before a value is returned.


In [ ]:
rolling = regular[["sales"]].copy()
rolling["mean_3"] = rolling["sales"].rolling(3, min_periods=2).mean()
rolling["mean_12"] = rolling["sales"].rolling(12, min_periods=6).mean()
rolling["std_6"] = rolling["sales"].rolling(6, min_periods=3).std()

print(rolling.tail().round(2))


### Plot raw and rolling series

The trailing mean smooths seasonal and irregular movement but also delays turning points.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rolling.index, rolling["sales"], label="Monthly", alpha=0.45)
ax.plot(rolling.index, rolling["mean_12"], label="12-month trailing mean", linewidth=2.5)
ax.set_title("Sales and trailing annual mean")
ax.set_ylabel("Sales (£000)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Resample to a lower frequency

Aggregation must match meaning:

- flows such as monthly sales are usually summed over a quarter;
- rates or prices may be averaged;
- stock values may use the final observation.


### Create quarterly totals and averages

Different columns receive aggregation rules suited to their meaning.


In [ ]:
quarterly = regular.resample("QS").agg(
    sales_total=("sales", "sum"),
    orders_total=("orders", "sum"),
    marketing_mean=("marketing_spend", "mean"),
)
print(quarterly.head().round(2))


### Create annual summaries

YE labels each group at year end. min_count avoids turning an entirely missing period into zero.


In [ ]:
annual_sales = regular["sales"].resample("YE").sum(min_count=1)
annual_orders = regular["orders"].resample("YE").sum(min_count=1)
annual_summary = pd.DataFrame({
    "sales_total": annual_sales,
    "orders_total": annual_orders,
})
print(annual_summary.round(1))


## Guided time-series audit

Before any trend statement, report start/end, frequency, number of periods, duplicate timestamps, missing periods, and missing values.


### Create an audit record

The audit makes temporal assumptions visible.


In [ ]:
time_audit = pd.Series({
    "start": regular.index.min(),
    "end": regular.index.max(),
    "frequency": pd.infer_freq(regular.index),
    "periods": len(regular),
    "duplicate_timestamps": regular.index.duplicated().sum(),
    "missing_sales": regular["sales"].isna().sum(),
    "completely_missing_periods": regular.isna().all(axis=1).sum(),
})
print(time_audit)


## Student coding lab

Preserve chronological order and do not use random train/test splitting for these tasks.


        ### Student task 1: Create a regular time index

        <div class="task"><strong>Your job:</strong> Import monthly_sales.csv, parse month, sort it, set the index, convert to MS frequency, and audit order, uniqueness, and missingness.</div>

        **Check your work**

        - The index is DatetimeIndex.
- Frequency is monthly start.
- Absent periods and missing cell values are counted separately.


In [ ]:
# STUDENT TASK 1
task_series = pd.DataFrame()
task_audit = {}
print(task_audit)


        ### Student task 2: Create lags and differences

        <div class="task"><strong>Your job:</strong> Add 1-month and 12-month sales lags plus first and seasonal differences.</div>

        **Check your work**

        - Lag direction uses past values.
- The 12-month comparison uses monthly frequency.
- Initial unavailable comparisons remain missing.


In [ ]:
# STUDENT TASK 2
lag_table = regular[["sales"]].copy()
# Add four requested columns.
print(lag_table.tail())


        ### Student task 3: Calculate growth safely

        <div class="task"><strong>Your job:</strong> Calculate month-on-month and year-on-year percentage growth without implicitly filling missing sales.</div>

        **Check your work**

        - Growth is multiplied by 100 for percentages.
- Missing inputs produce missing growth.
- Periods used in each comparison are stated.


In [ ]:
# STUDENT TASK 3
growth_table = regular[["sales"]].copy()
# Add both growth columns with fill_method=None.
print(growth_table.tail(15))


        ### Student task 4: Compare rolling windows

        <div class="task"><strong>Your job:</strong> Calculate 3-, 6-, and 12-month trailing sales means, then plot all three with raw sales.</div>

        **Check your work**

        - All windows trail rather than centre.
- Longer windows appear smoother.
- A comment notes lag and missing-value effects.


In [ ]:
# STUDENT TASK 4
rolling_comparison = regular[["sales"]].copy()
fig, ax = plt.subplots(figsize=(9, 4))
# Calculate and plot all series.
plt.close(fig)
print(rolling_comparison.tail())


        ### Student task 5: Resample with meaningful rules

        <div class="task"><strong>Your job:</strong> Create quarterly and annual summaries using sum for sales/orders and mean for marketing spend. Explain the choices.</div>

        **Check your work**

        - Each column uses a stated aggregation rule.
- The index remains datetime.
- The explanation distinguishes flows from average intensity.



**Optional extension:** Compare quarter-start and quarter-end labels and explain their representational difference.


In [ ]:
# STUDENT TASK 5
task_quarterly = pd.DataFrame()
task_annual = pd.DataFrame()
print(task_quarterly.head())
print(task_annual)
# Explanation:


        ## Knowledge check

        Answer these without running new code first.

        1. Why must time be sorted?
2. What does a 12-period lag mean at monthly frequency?
3. Why is the first difference missing?
4. How can a rolling mean lag a turning point?
5. Why do columns need different resampling rules?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 compares growth rates, interpolation, frequency changes, seasonality, and time-aware validation splits.
